# Phase 2 — Risk Ground-Truth Label Audit (V1)
**Independent, read-only evaluation** of the Schedule, Dependency and Scope labels. Based on `GroundTruth_TemporalSnapshots(1).ipynb` and `Feature_Engineering.ipynb`.

**Known rules from teammate source:** 14-day future outcome horizon; positive when **at least 3 distinct affected issues** occur in the horizon. The event eligibility filters are more involved and are **not reconstructed by this audit**. Passing a threshold check does **not** prove that underlying events are valid, nor that labels are independent of model features.

Expected: 987 checkpoints across 30 project instances. All audit outputs go to `MyDrive/02_Risk_Prediction/Label_Validation/Results/V1/`. Upstream data are never modified.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, numpy as np, json, hashlib
ROOT=Path('/content/drive/MyDrive/GC1/Temporal_snapshots')
OUT=Path('/content/drive/MyDrive/02_Risk_Prediction/Label_Validation/Results/V1')
OUT.mkdir(parents=True,exist_ok=True)
print('Source:',ROOT,'\nAudit output:',OUT)

Mounted at /content/drive
Source: /content/drive/MyDrive/GC1/Temporal_snapshots 
Audit output: /content/drive/MyDrive/02_Risk_Prediction/Label_Validation/Results/V1


## 1. Locate canonical label exports
Execution stops when a required label export is absent. It never silently substitutes an unrelated experimental file.

In [2]:
FILES={
 'schedule': ROOT/'schedule_risk_labels.csv',
 'dependency': ROOT/'dependency_escalation_labels.csv',
 'scope': ROOT/'scope_risk_labels.csv',
 'combined': ROOT/'phase2_three_risk_labels.csv',
 'features': ROOT/'feature_engineering_output'/'tabpfn_labelled_dataset.parquet',
}
for k,p in FILES.items():print(k,'FOUND' if p.exists() else 'MISSING',p)
required=['schedule','dependency','scope','combined']
missing=[k for k in required if not FILES[k].exists()]
assert not missing, f'Missing canonical inputs: {missing}. Check the teammate exports and ROOT; do not guess alternative files.'
frames={k:pd.read_csv(FILES[k]) for k in required}
for k,d in frames.items():print(k,len(d),list(d.columns))

schedule FOUND /content/drive/MyDrive/GC1/Temporal_snapshots/schedule_risk_labels.csv
dependency FOUND /content/drive/MyDrive/GC1/Temporal_snapshots/dependency_escalation_labels.csv
scope FOUND /content/drive/MyDrive/GC1/Temporal_snapshots/scope_risk_labels.csv
combined FOUND /content/drive/MyDrive/GC1/Temporal_snapshots/phase2_three_risk_labels.csv
features FOUND /content/drive/MyDrive/GC1/Temporal_snapshots/feature_engineering_output/tabpfn_labelled_dataset.parquet
schedule 987 ['instance_id', 'week_number', 'checkpoint_time', 'horizon_end', 'future_schedule_events', 'future_affected_issues', 'future_reopened_issues', 'future_regressed_issues', 'future_blocked_issues', 'schedule_risk_label']
dependency 987 ['instance_id', 'week_number', 'checkpoint_time', 'horizon_end', 'future_new_links', 'future_affected_downstream', 'dependency_risk_label']
scope 987 ['instance_id', 'week_number', 'checkpoint_time', 'horizon_end', 'future_scope_events', 'future_affected_issues', 'future_epic_child

## 2. Validate keys, label definitions and time windows

In [3]:
KEYS=['instance_id','week_number','checkpoint_time']
LABELS={'schedule':'schedule_risk_label','dependency':'dependency_risk_label','scope':'scope_risk_label'}
checks=[]
def check(stage,test,passed,detail):checks.append(dict(stage=stage,test=test,status='PASS' if bool(passed) else 'FAIL',detail=str(detail)))
def dates(d):
 for c in ['checkpoint_time','horizon_end']:
  if c in d:d[c]=pd.to_datetime(d[c],utc=True,errors='coerce',format='mixed')
 return d
for k,d in frames.items():
 dates(d)
 for col in KEYS:assert col in d, f'{k} missing {col}'
 check(k,'Unique checkpoint key',not d.duplicated(KEYS).any(),int(d.duplicated(KEYS).sum()))
 check(k,'Parseable checkpoint time',d.checkpoint_time.notna().all(),int(d.checkpoint_time.isna().sum()))
 check(k,'30 instances',d.instance_id.nunique()==30,d.instance_id.nunique())
 check(k,'987 checkpoints',len(d)==987,len(d))
 if k in LABELS:
  y=LABELS[k];assert y in d, f'{k}: missing {y}'
  check(k,'Binary complete labels',d[y].notna().all() and d[y].isin([0,1]).all(),d[y].value_counts(dropna=False).to_dict())
  assert 'future_affected_issues' in d if k!='dependency' else 'future_affected_downstream' in d, f'{k}: missing future outcome count'
  countcol='future_affected_downstream' if k=='dependency' else 'future_affected_issues'
  expected=(pd.to_numeric(d[countcol],errors='coerce')>=3).astype(int)
  check(k,'Threshold >=3 affected issues',(d[y]==expected).all(),int((d[y]!=expected).sum()))
  if 'horizon_end' in d:
   check(k,'Exactly 14-day horizon',((d.horizon_end-d.checkpoint_time)==pd.Timedelta(days=14)).all(),int(((d.horizon_end-d.checkpoint_time)!=pd.Timedelta(days=14)).sum()))
  else:check(k,'14-day horizon column available',False,'horizon_end absent; cannot check')
checks_df=pd.DataFrame(checks);display(checks_df)
checks_df.to_csv(OUT/'label_rule_checks.csv',index=False)

,stage,test,status,detail
0,schedule,Unique checkpoint key,PASS,0
1,schedule,Parseable checkpoint time,PASS,0
2,schedule,30 instances,PASS,30
3,schedule,987 checkpoints,PASS,987
4,schedule,Binary complete labels,PASS,"{0: 722, 1: 265}"
5,schedule,Threshold >=3 affected issues,PASS,0
6,schedule,Exactly 14-day horizon,PASS,0
7,dependency,Unique checkpoint key,PASS,0
8,dependency,Parseable checkpoint time,PASS,0
9,dependency,30 instances,PASS,30


## 3. Check that the consolidated labels match all three source tables exactly

In [4]:
combined=frames['combined']; merge=combined[KEYS].copy()
for k in ['schedule','dependency','scope']:
 d=frames[k]; y=LABELS[k]
 merge=merge.merge(d[KEYS+[y]],on=KEYS,how='left',validate='one_to_one',suffixes=('','_source'))
 mismatches=(merge[y].isna() | (merge[y]!=combined[y])).sum()
 check('combined',f'{y} matches individual export',mismatches==0,int(mismatches))
print('Consolidated mismatch checks:');display(pd.DataFrame(checks).tail(3))
pd.DataFrame(checks).to_csv(OUT/'label_rule_checks.csv',index=False)

Consolidated mismatch checks:


,stage,test,status,detail
25,combined,schedule_risk_label matches individual export,PASS,0
26,combined,dependency_risk_label matches individual export,PASS,0
27,combined,scope_risk_label matches individual export,PASS,0


## 4. Class balance and instance-level concentration
High scores on labels concentrated in a few projects can be misleading.

In [5]:
rows=[]
for k,y in LABELS.items():
 d=frames[k]
 for inst,g in d.groupby('instance_id'):
  rows.append({'risk':k,'instance_id':inst,'checkpoints':len(g),'positive':int(g[y].sum()),'positive_rate':float(g[y].mean())})
balance=pd.DataFrame(rows)
display(balance.groupby('risk').agg(checkpoints=('checkpoints','sum'),positives=('positive','sum'),positive_instances=('positive',lambda s:int((s>0).sum()))))
balance.to_csv(OUT/'risk_label_distribution_by_instance.csv',index=False)

,checkpoints,positives,positive_instances
risk,,,
dependency,987,88,10
schedule,987,265,24
scope,987,167,20


## 5. Feature/label leakage screening
This is a **column-name and key-alignment screen**, not proof of absence of all temporal leakage. Feature formulas and training splits require a separate audit.

In [6]:
if FILES['features'].exists():
 f=pd.read_parquet(FILES['features']);print('Feature rows:',len(f),'columns:',len(f.columns))
 future=[c for c in f.columns if c.lower().startswith('future_') or c.lower().startswith('horizon_')]
 target=[c for c in f.columns if c in LABELS.values()]
 print('Future-outcome columns in labelled table:',future)
 print('Target columns (expected as targets, NOT model features):',target)
 check('features','No future outcome columns in labelled matrix',not future,future)
 if all(x in f for x in KEYS):
  dates(f)
  check('features','Feature keys unique',not f.duplicated(KEYS).any(),int(f.duplicated(KEYS).sum()))
 else:check('features','Feature keys present',False,'Missing checkpoint keys')
else:
 print('Feature matrix not present: leakage screening skipped (not a PASS).')
 check('features','Feature matrix accessible',False,'Not found; cannot assess')
pd.DataFrame(checks).to_csv(OUT/'label_rule_checks.csv',index=False)

Feature rows: 987 columns: 40
Future-outcome columns in labelled table: []
Target columns (expected as targets, NOT model features): ['schedule_risk_label', 'scope_risk_label', 'dependency_risk_label']


## 6. Stratified review sample (optional human evidence audit)
Samples positive and negative checkpoints per risk across instances. The columns for human assessment remain blank. **Do not call these human-validated until someone checks source events.**

In [7]:
sample=[]
for k,y in LABELS.items():
 d=frames[k].copy();d['risk_type']=k;d['label']=d[y]
 for label,g in d.groupby('label'):
  # reproducible broad coverage by sampling at most one per instance, then filling to 15
  candidates=g.sort_values(['instance_id','checkpoint_time']).groupby('instance_id',group_keys=False).head(1)
  chosen=candidates.sample(n=min(15,len(candidates)),random_state=42)
  if len(chosen)<min(15,len(g)):
   remaining=g.drop(chosen.index).sample(n=min(15-len(chosen),len(g)-len(chosen)),random_state=43)
   chosen=pd.concat([chosen,remaining])
  sample.append(chosen)
review=pd.concat(sample,ignore_index=True)
review['source_evidence_checked']=''
review['reviewer_label']=''
review['reviewer_notes']=''
review['review_status']='NOT_REVIEWED'
review.to_csv(OUT/'stratified_label_review_template.csv',index=False)
print('Sample size:',len(review));display(review[['risk_type','label']].value_counts().rename('n'))

Sample size: 90


risk_type   label
dependency  0        15
            1        15
schedule    0        15
            1        15
scope       0        15
            1        15
Name: n, dtype: int64

## 7. Export audit summary and interpretation
**PASS** on rules means exported labels follow the stated rules. It does not establish whether the chosen thresholds define meaningful real-world risk or whether future labels were accidentally used as input features.

In [8]:
checks_df=pd.DataFrame(checks)
summary={'risk_labels':list(LABELS.values()),'horizon_days':14,'affected_issue_threshold':3,
 'checks_passed':int((checks_df.status=='PASS').sum()),'checks_failed_or_unavailable':int((checks_df.status!='PASS').sum()),
 'label_rows':{k:len(frames[k]) for k in frames},
 'limitations':['Threshold consistency is not construct validity.','Event eligibility and original-source truth are not independently reconstructed.','Feature-column screen is not full leakage testing.','Project-grouped split and predictive model scores require separate audit.','Sample annotations are blank until independently reviewed.']}
(OUT/'label_audit_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2));print('Saved reports:',OUT)
if summary['checks_failed_or_unavailable']:print('ATTENTION: Review failed/unavailable checks before freezing labels.')

{
  "risk_labels": [
    "schedule_risk_label",
    "dependency_risk_label",
    "scope_risk_label"
  ],
  "horizon_days": 14,
  "affected_issue_threshold": 3,
  "checks_passed": 30,
  "checks_failed_or_unavailable": 0,
  "label_rows": {
    "schedule": 987,
    "dependency": 987,
    "scope": 987,
    "combined": 987
  },
  "limitations": [
    "Threshold consistency is not construct validity.",
    "Event eligibility and original-source truth are not independently reconstructed.",
    "Feature-column screen is not full leakage testing.",
    "Project-grouped split and predictive model scores require separate audit.",
    "Sample annotations are blank until independently reviewed."
  ]
}
Saved reports: /content/drive/MyDrive/02_Risk_Prediction/Label_Validation/Results/V1
